# Private RSNA lossless cache part 2/2

Technical preprocessing only. Keep this notebook and its outputs private. No cancer-model evaluation is performed. Storage partitions are independent of evaluation folds.

In [ ]:
from pathlib import Path
import sys
import subprocess
PROJECT = Path('/kaggle/working/cache_project')
PROJECT.mkdir(exist_ok=True)
(PROJECT / 'mammo_reliability').mkdir(exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'pydicom>=3,<4', 'pylibjpeg>=2,<3', 'pylibjpeg-openjpeg>=2,<3', 'pylibjpeg-libjpeg>=2,<3'], check=True)


In [ ]:
SOURCE = '"""Research software. No clinical validation is implied."""\n\n__version__ = "0.1.0"\n'
(PROJECT / "mammo_reliability" / '__init__.py').write_text(SOURCE)


In [ ]:
SOURCE = '"""Validate original RSNA metadata without silently changing the cohort."""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold, train_test_split\n\nBREAST_KEY = ["patient_id", "laterality"]\nSTANDARD_VIEWS = ["CC", "MLO"]\nREQUIRED = ["site_id", "patient_id", "image_id", "laterality", "view", "cancer", "density"]\nOUTCOME_ONLY = ["cancer", "biopsy", "invasive", "BIRADS", "difficult_negative_case"]\n\n\ndef require(condition: bool, message: str) -> None:\n    """Unlike assert, this check is not disabled by Python\'s -O option."""\n    if not bool(condition):\n        raise ValueError(message)\n\n\ndef file_sha256(path: str | Path) -> str:\n    h = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef environment() -> dict:\n    packages = {}\n    for name in ["numpy", "pandas", "scipy", "scikit-learn", "Pillow", "pydicom",\n                 "torch", "torchvision", "pylibjpeg", "pylibjpeg-openjpeg"]:\n        try:\n            packages[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            packages[name] = None\n    return {"python": platform.python_version(), "platform": platform.platform(), "packages": packages}\n\n\ndef write_json(path: str | Path, payload: dict) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    temp.write_text(json.dumps(payload, indent=2, allow_nan=False) + "\\n")\n    temp.replace(path)\n\n\ndef locate_dataset() -> Path:\n    candidates = [Path("/kaggle/input/competitions/rsna-breast-cancer-detection"),\n                  Path("/kaggle/input/rsna-breast-cancer-detection")]\n    for root in candidates:\n        if (root / "train.csv").is_file():\n            return root\n    raise FileNotFoundError("Attach the official RSNA competition dataset or provide its local root explicitly.")\n\n\ndef audit_metadata(images: pd.DataFrame) -> tuple[pd.DataFrame, dict, dict[str, pd.DataFrame]]:\n    require(len(images) > 0, "Metadata is empty.")\n    require(set(REQUIRED).issubset(images.columns), f"Missing columns: {set(REQUIRED) - set(images.columns)}")\n    for col in ["site_id", "patient_id", "image_id", "laterality", "view"]:\n        require(images[col].notna().all(), f"Missing identifying information in {col}.")\n    for col in ["site_id", "patient_id", "image_id"]:\n        values = pd.to_numeric(images[col], errors="coerce")\n        require((values.notna() & np.isfinite(values) & (values >= 0) & (values == np.floor(values))).all(),\n                f"{col} must contain nonnegative integer identifiers.")\n    require(images["image_id"].is_unique, "Repeated image IDs require investigation.")\n    require(images["laterality"].isin(["L", "R"]).all(), "Unexpected laterality.")\n    require(images["cancer"].isin([0, 1]).all(), "Missing or nonbinary cancer labels.")\n    require((images.groupby("patient_id")["site_id"].nunique() == 1).all(), "A patient occurs at multiple sites.")\n    grouped = images.groupby(BREAST_KEY, sort=True)\n    require((grouped["cancer"].nunique(dropna=False) == 1).all(), "Cancer labels conflict within a breast.")\n    observed_density = images["density"].dropna()\n    require(observed_density.isin(list("ABCD")).all(), "Unexpected density category; inspect the source values.")\n\n    breasts = grouped.agg(site_id=("site_id", "first"), cancer=("cancer", "first"),\n                          n_images=("image_id", "size"), density=("density", "first"),\n                          n_density_labels=("density", "count"),\n                          n_density_values=("density", "nunique"))\n    views = pd.crosstab(index=[images[c] for c in BREAST_KEY], columns=images["view"])\n    views = views.reindex(index=breasts.index, columns=STANDARD_VIEWS, fill_value=0)\n    breasts["n_cc"] = views["CC"]\n    breasts["n_mlo"] = views["MLO"]\n    breasts["has_both_views"] = (breasts["n_cc"] > 0) & (breasts["n_mlo"] > 0)\n    breasts["density_missing"] = breasts["n_density_labels"] == 0\n    breasts["density_partly_missing"] = ((breasts["n_density_labels"] > 0) &\n                                           (breasts["n_density_labels"] < breasts["n_images"]))\n    breasts["density_conflict"] = breasts["n_density_values"] > 1\n    # The first available value is retained only if all available values agree.\n    breasts.loc[breasts["density_conflict"], "density"] = np.nan\n    breasts["density_group"] = breasts["density"].map({"A": "A/B", "B": "A/B", "C": "C/D", "D": "C/D"})\n    breasts["has_extra_images"] = breasts["n_images"] > 2\n    if "age" in images:\n        breasts["age"] = grouped["age"].first()\n        breasts["age_conflict"] = grouped["age"].nunique() > 1\n        breasts.loc[breasts["age_conflict"], "age"] = np.nan\n    breasts = breasts.reset_index()\n\n    extra = breasts.groupby(["site_id", "cancer"], as_index=False).agg(\n        breast_count=("cancer", "size"), breasts_with_extra_images=("has_extra_images", "sum"),\n        extra_image_fraction=("has_extra_images", "mean"))\n    density = breasts.groupby("site_id", as_index=False).agg(\n        breast_count=("cancer", "size"), all_missing=("density_missing", "sum"),\n        partly_missing=("density_partly_missing", "sum"), conflicts=("density_conflict", "sum"))\n    tables = {\n        "site_cancer": breasts.groupby(["site_id", "cancer"]).size().rename("breast_count").reset_index(),\n        "extra_images": extra, "density_availability": density,\n        "density_cancer": breasts.assign(density=breasts["density"].fillna("Missing/conflicting"))\n            .groupby(["site_id", "density", "cancer"]).size().rename("breast_count").reset_index(),\n        "image_views": images["view"].value_counts(dropna=False).rename_axis("view").rename("image_count").reset_index(),\n        "missing_columns": images.isna().sum().rename_axis("column").rename("missing_image_rows").reset_index(),\n    }\n    summary = {\n        "source": "computed_from_supplied_metadata", "image_rows": int(len(images)),\n        "patients": int(images["patient_id"].nunique()), "breasts": int(len(breasts)),\n        "positive_breasts": int(breasts["cancer"].sum()),\n        "positive_patients": int(breasts.groupby("patient_id")["cancer"].max().sum()),\n        "positive_breast_fraction": float(breasts["cancer"].mean()),\n        "breasts_missing_standard_view": int((~breasts["has_both_views"]).sum()),\n        "breasts_with_extra_images": int(breasts["has_extra_images"].sum()),\n        "density_conflicts": int(breasts["density_conflict"].sum()),\n        "pixel_duplicates_checked": False,\n    }\n    return breasts, summary, tables\n\n\ndef make_patient_folds(breasts: pd.DataFrame, n_splits: int = 5, seed: int = 20260909) -> pd.DataFrame:\n    require(breasts[BREAST_KEY].duplicated().sum() == 0, "Expected one row per breast.")\n    require(breasts["has_both_views"].all(), "Resolve breasts missing CC or MLO before defining the paired cohort.")\n    patients = breasts.groupby("patient_id", as_index=False).agg(site_id=("site_id", "first"),\n                                                               patient_cancer=("cancer", "max"))\n    strata = patients["site_id"].astype(str) + ":" + patients["patient_cancer"].astype(str)\n    require(strata.value_counts().min() >= n_splits, "Insufficient patients per site/cancer stratum for these folds.")\n    patients["outer_fold"] = -1\n    for fold, (_, test_idx) in enumerate(StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(patients, strata)):\n        patients.loc[test_idx, "outer_fold"] = fold\n    require((patients["outer_fold"] >= 0).all(), "Some patients lack a fold.")\n    return patients\n\n\ndef partition_outer_fold(patients: pd.DataFrame, fold: int, seed: int = 20260909) -> pd.DataFrame:\n    require(patients["patient_id"].is_unique, "Split input must contain unique patients.")\n    require(fold in patients["outer_fold"].unique(), "Unknown outer fold.")\n    test = patients["outer_fold"] == fold\n    dev = patients.loc[~test].copy()\n    strata = dev["site_id"].astype(str) + ":" + dev["patient_cancer"].astype(str)\n    # Fractions of the complete cohort: fit 60%, tune 10%, calibrate 10%, test 20%.\n    fit_idx, held_idx = train_test_split(dev.index, test_size=0.25, stratify=strata, random_state=seed + fold)\n    held = dev.loc[held_idx]\n    held_strata = held["site_id"].astype(str) + ":" + held["patient_cancer"].astype(str)\n    tune_idx, cal_idx = train_test_split(held.index, test_size=0.5, stratify=held_strata, random_state=seed + 100 + fold)\n    result = patients.copy()\n    result["role"] = "test"\n    result.loc[fit_idx, "role"] = "fit"\n    result.loc[tune_idx, "role"] = "tune"\n    result.loc[cal_idx, "role"] = "calibrate"\n    require((result.loc[test, "role"] == "test").all(), "Outer test patients leaked into development.")\n    require((result.loc[~test, "role"] != "test").all(), "Development patient role was not assigned.")\n    for role, frame in result.groupby("role"):\n        require(set(frame["patient_cancer"].unique()) == {0, 1}, f"Role {role} lacks a class.")\n    return result\n'
(PROJECT / "mammo_reliability" / 'audit.py').write_text(SOURCE)


In [ ]:
SOURCE = '"""DICOM presentation processing, preserving the full field of view."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport pydicom\nfrom pydicom.pixels import apply_modality_lut, apply_voi_lut\n\nfrom .audit import STANDARD_VIEWS, file_sha256, require, write_json\n\n\ndef normalize_pixels(raw: np.ndarray, presented: np.ndarray, photometric: str,\n                     padding_value=None, padding_limit=None) -> np.ndarray:\n    require(raw.ndim == 2 and presented.shape == raw.shape, "Only two-dimensional grayscale images are supported.")\n    require(photometric in ["MONOCHROME1", "MONOCHROME2"], "Unsupported photometric interpretation.")\n    valid = np.isfinite(presented)\n    if padding_value is not None:\n        if padding_limit is None:\n            valid &= raw != padding_value\n        else:\n            lower, upper = sorted([padding_value, padding_limit])\n            valid &= ~((raw >= lower) & (raw <= upper))\n    require(valid.any(), "No valid image pixels remain.")\n    low, high = float(presented[valid].min()), float(presented[valid].max())\n    require(high > low, "Uniform image after presentation processing.")\n    result = ((presented.astype(np.float32) - low) / (high - low)).clip(0, 1)\n    if photometric == "MONOCHROME1":\n        result = 1 - result\n    result[~valid] = 0\n    return result.astype(np.float32)\n\n\ndef letterbox(array: np.ndarray, height: int = 1024, width: int = 768) -> np.ndarray:\n    require(array.ndim == 2 and min(height, width) > 0, "Invalid image or target dimensions.")\n    h, w = array.shape\n    factor = min(height / h, width / w)\n    new_h, new_w = max(1, round(h * factor)), max(1, round(w * factor))\n    resized = np.asarray(Image.fromarray(array.astype(np.float32)).resize(\n        (new_w, new_h), resample=Image.Resampling.BILINEAR))\n    canvas = np.zeros((height, width), dtype=np.float32)\n    y, x = (height - new_h) // 2, (width - new_w) // 2\n    canvas[y:y + new_h, x:x + new_w] = resized\n    return canvas\n\n\ndef decode_dicom(path: str | Path, height: int = 1024, width: int = 768) -> tuple[np.ndarray, dict]:\n    ds = pydicom.dcmread(path)\n    raw = ds.pixel_array\n    if raw.ndim == 3 and raw.shape[0] == 1:\n        raw = raw[0]\n    require(raw.ndim == 2, "Multiframe/color DICOM requires a separate preprocessing decision.")\n    require("PresentationLUTSequence" not in ds, "Presentation LUT sequence requires image-specific review.")\n    photo = str(ds.get("PhotometricInterpretation", ""))\n    presented = apply_voi_lut(apply_modality_lut(raw, ds), ds)\n    normalized = normalize_pixels(raw, presented, photo, ds.get("PixelPaddingValue"), ds.get("PixelPaddingRangeLimit"))\n    digest = hashlib.sha256()\n    digest.update(str((raw.shape, str(raw.dtype))).encode())\n    digest.update(np.ascontiguousarray(raw).tobytes())\n    info = {"height_original": int(raw.shape[0]), "width_original": int(raw.shape[1]),\n            "photometric": photo, "decoded_pixel_sha256": digest.hexdigest(),\n            "voi_present": bool("VOILUTSequence" in ds or "WindowCenter" in ds),\n            "nonzero_fraction_after_normalization": float((normalized > 0).mean())}\n    return letterbox(normalized, height, width), info\n\n\ndef check_cross_patient_duplicates(manifest: pd.DataFrame) -> None:\n    require("decoded_pixel_sha256" in manifest, "Pixel duplicate audit was not performed.")\n    shared = manifest.groupby("decoded_pixel_sha256")["patient_id"].nunique()\n    require(not (shared > 1).any(), "Identical decoded images occur under different patients. Investigate/group them before training.")\n\n\ndef prepare_images(images: pd.DataFrame, data_root: str | Path, output: str | Path,\n                   height: int = 1024, width: int = 768, cache_images: bool = True) -> pd.DataFrame:\n    """Resume verified 16-bit PNG conversion, or audit pixels without caching.\n\n    Never skip failed images. A stopped run can resume from the last manifest.\n    An explicit disk check after the first 32 images prevents filling the volume.\n    """\n    data_root, output = Path(data_root), Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    source_hash = file_sha256(data_root / "train.csv")\n    config = {"version": 1, "height": height, "width": width, "cache_images": cache_images, "metadata_sha256": source_hash}\n    config_file, manifest_file = output / "preprocessing.json", output / "manifest.csv"\n    if config_file.exists():\n        require(json.loads(config_file.read_text()) == config, "Preprocessing settings/source changed; use a new output directory.")\n    else:\n        write_json(config_file, config)\n    rows = images.loc[images["view"].isin(STANDARD_VIEWS)].sort_values(["patient_id", "image_id"])\n    previous = pd.read_csv(manifest_file).to_dict("records") if manifest_file.exists() else []\n    recorded = {int(r["image_id"]): r for r in previous}\n    require(len(recorded) == len(previous), "Repeated image in preprocessing manifest.")\n    require(set(recorded).issubset(set(rows["image_id"].astype(int))), "Manifest has unexpected images.")\n    cached_sizes = []\n    for row in rows.itertuples(index=False):\n        image_id, patient_id = int(row.image_id), int(row.patient_id)\n        if image_id in recorded:\n            old = recorded[image_id]\n            if cache_images:\n                require(Path(old["cache_path"]).is_file(), "A recorded cached image is missing.")\n                require(file_sha256(old["cache_path"]) == old["cache_sha256"], "Cached image hash mismatch.")\n                cached_sizes.append(Path(old["cache_path"]).stat().st_size)\n            continue\n        source = data_root / "train_images" / str(patient_id) / f"{image_id}.dcm"\n        array, info = decode_dicom(source, height, width)\n        record = {"patient_id": patient_id, "image_id": image_id, "laterality": row.laterality,\n                  "view": row.view, "source_path": str(source), **info}\n        if cache_images:\n            cache_path = output / "images" / str(patient_id) / f"{image_id}.png"\n            cache_path.parent.mkdir(parents=True, exist_ok=True)\n            Image.fromarray(np.round(array * 65535).astype(np.uint16)).save(cache_path)\n            record.update(cache_path=str(cache_path), cache_sha256=file_sha256(cache_path))\n            cached_sizes.append(cache_path.stat().st_size)\n        recorded[image_id] = record\n        if len(recorded) % 32 == 0 or len(recorded) == len(rows):\n            temporary = manifest_file.with_suffix(".csv.tmp")\n            pd.DataFrame(recorded.values()).to_csv(temporary, index=False)\n            temporary.replace(manifest_file)\n            print(f"Pixel audit: {len(recorded)}/{len(rows)} images", flush=True)\n            if cache_images:\n                estimate = float(np.mean(cached_sizes)) * (len(rows) - len(recorded)) * 1.25\n                require(estimate + 2 * 1024**3 < shutil.disk_usage(output).free,\n                        "Estimated PNG cache exceeds free disk space. Use a larger volume, or a NEW output directory with cache_images=False.")\n    manifest = pd.DataFrame(recorded.values()).sort_values(["patient_id", "image_id"]).reset_index(drop=True)\n    require(len(manifest) == len(rows), "Incomplete image manifest.")\n    check_cross_patient_duplicates(manifest)\n    write_json(output / "pixel_audit_complete.json", {"metadata_sha256": source_hash, "n_images": len(manifest),\n                                                      "manifest_sha256": file_sha256(manifest_file),\n                                                      "cross_patient_exact_pixel_duplicates": 0})\n    return manifest\n'
(PROJECT / "mammo_reliability" / 'imaging.py').write_text(SOURCE)


In [ ]:
SOURCE = '"""Build private lossless cache parts, then validate and mount them together.\n\nThe partition is only for storage. It does not change patient evaluation folds.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom concurrent.futures import ProcessPoolExecutor\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom .audit import STANDARD_VIEWS, file_sha256, locate_dataset, require, write_json\nfrom .imaging import check_cross_patient_duplicates, prepare_images\n\n\ndef preprocessing_fingerprint() -> str:\n    return file_sha256(Path(__file__).with_name(\'imaging.py\'))\n\n\ndef _prepare_shard(task):\n    images, data_root, folder, height, width = task\n    return prepare_images(images, data_root, folder, height, width, True)\n\n\ndef build_part(data_root: Path, output: Path, part: int, n_parts: int = 2,\n               workers: int = 4, height: int = 1024, width: int = 768) -> dict:\n    require(n_parts > 0 and 0 <= part < n_parts and workers > 0, \'Invalid cache partition settings.\')\n    images = pd.read_csv(data_root / \'train.csv\')\n    images = images.loc[images[\'view\'].isin(STANDARD_VIEWS)].copy()\n    selected = images.loc[images[\'patient_id\'] % n_parts == part].copy()\n    require(len(selected) > 0, \'Empty cache part.\')\n    identity = {\'version\': 1, \'part\': part, \'n_parts\': n_parts, \'workers\': workers,\n                \'height\': height, \'width\': width, \'metadata_sha256\': file_sha256(data_root / \'train.csv\'),\n                \'image_processing_source_sha256\': preprocessing_fingerprint(),\n                \'n_images\': len(selected), \'cache_format\': \'16-bit PNG after full-frame letterbox\'}\n    output.mkdir(parents=True, exist_ok=True)\n    identity_file = output / \'part_identity.json\'\n    if identity_file.exists():\n        require(json.loads(identity_file.read_text()) == identity, \'Cache part settings or preprocessing changed.\')\n    else:\n        write_json(identity_file, identity)\n    tasks = []\n    shard_id = (selected[\'patient_id\'] // n_parts) % workers\n    for shard in range(workers):\n        subset = selected.loc[shard_id == shard]\n        if len(subset):\n            tasks.append((subset, data_root, output / f\'shard_{shard}\', height, width))\n    print(f\'Preparing private cache part {part + 1}/{n_parts}: {len(selected)} images; {len(tasks)} workers.\', flush=True)\n    if workers == 1:\n        frames = [_prepare_shard(task) for task in tasks]\n    else:\n        with ProcessPoolExecutor(max_workers=workers) as pool:\n            frames = list(pool.map(_prepare_shard, tasks))\n    manifest = pd.concat(frames, ignore_index=True).sort_values([\'patient_id\', \'image_id\'])\n    require(manifest[\'image_id\'].is_unique, \'Repeated image in cache part.\')\n    require(set(manifest[\'image_id\']) == set(selected[\'image_id\']), \'Cache part image coverage differs from metadata.\')\n    check_cross_patient_duplicates(manifest)\n    manifest[\'cache_relative_path\'] = manifest[\'cache_path\'].map(lambda p: str(Path(p).relative_to(output)))\n    manifest.to_csv(output / \'manifest.csv\', index=False)\n    result = {**identity, \'manifest_sha256\': file_sha256(output / \'manifest.csv\'),\n              \'cross_patient_exact_pixel_duplicates_within_part\': 0}\n    write_json(output / \'part_complete.json\', result)\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\n\ndef assemble_parts(part_roots: list[Path], data_root: Path, output: Path,\n                   height: int = 1024, width: int = 768) -> pd.DataFrame:\n    require(len(part_roots) > 0, \'Attach the completed private cache parts first.\')\n    metadata_hash = file_sha256(data_root / \'train.csv\')\n    frames, completions = [], []\n    for root in part_roots:\n        completion = json.loads((root / \'part_complete.json\').read_text())\n        require(completion[\'metadata_sha256\'] == metadata_hash, \'A cache part uses different source metadata.\')\n        require(completion[\'height\'] == height and completion[\'width\'] == width, \'Cache part resolution mismatch.\')\n        require(completion[\'image_processing_source_sha256\'] == preprocessing_fingerprint(), \'Cache preprocessing source mismatch.\')\n        require(completion[\'manifest_sha256\'] == file_sha256(root / \'manifest.csv\'), \'A cache part manifest changed.\')\n        frame = pd.read_csv(root / \'manifest.csv\')\n        require(len(frame) == completion[\'n_images\'], \'Incomplete cache part.\')\n        require((frame[\'patient_id\'] % completion[\'n_parts\'] == completion[\'part\']).all(), \'Cache contains the wrong patients.\')\n        def remount(relative):\n            path = Path(relative)\n            require(not path.is_absolute() and \'..\' not in path.parts, \'Unsafe relative cache path.\')\n            target = root / path\n            require(target.is_file(), \'A cached image is missing from the mounted input.\')\n            return str(target)\n        frame[\'cache_path\'] = frame[\'cache_relative_path\'].map(remount)\n        frame[\'source_path\'] = [str(data_root / \'train_images\' / str(int(p)) / f\'{int(i)}.dcm\')\n                                for p, i in zip(frame[\'patient_id\'], frame[\'image_id\'])]\n        frames.append(frame)\n        completions.append(completion)\n    expected_parts = completions[0][\'n_parts\']\n    require(all(c[\'n_parts\'] == expected_parts for c in completions), \'Incompatible cache partition counts.\')\n    require(len(completions) == expected_parts and {c[\'part\'] for c in completions} == set(range(expected_parts)),\n            \'Duplicate or missing cache parts.\')\n    manifest = pd.concat(frames, ignore_index=True).sort_values([\'patient_id\', \'image_id\']).reset_index(drop=True)\n    images = pd.read_csv(data_root / \'train.csv\')\n    expected = images.loc[images[\'view\'].isin(STANDARD_VIEWS)]\n    require(manifest[\'image_id\'].is_unique and set(manifest[\'image_id\']) == set(expected[\'image_id\']),\n            \'Combined cache does not cover every standard-view image exactly once.\')\n    keys = [\'patient_id\', \'image_id\', \'laterality\', \'view\']\n    require(manifest[keys].sort_values(\'image_id\').reset_index(drop=True).equals(\n        expected[keys].sort_values(\'image_id\').reset_index(drop=True)), \'Cache image identifiers or views mismatch metadata.\')\n    check_cross_patient_duplicates(manifest)\n    output.mkdir(parents=True, exist_ok=True)\n    manifest.to_csv(output / \'manifest.csv\', index=False)\n    write_json(output / \'preprocessing.json\', {\'version\': 1, \'height\': height, \'width\': width,\n               \'cache_images\': True, \'metadata_sha256\': metadata_hash})\n    write_json(output / \'pixel_audit_complete.json\', {\'metadata_sha256\': metadata_hash,\n               \'n_images\': len(manifest), \'manifest_sha256\': file_sha256(output / \'manifest.csv\'),\n               \'cross_patient_exact_pixel_duplicates\': 0, \'cache_part_completions\': completions})\n    print(f\'Validated and mounted {len(manifest)} images from {expected_parts} private parts.\', flush=True)\n    return manifest\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'stage\', choices=[\'build\', \'assemble\'])\n    parser.add_argument(\'--data-root\', type=Path)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--part\', type=int, default=0)\n    parser.add_argument(\'--n-parts\', type=int, default=2)\n    parser.add_argument(\'--workers\', type=int, default=4)\n    parser.add_argument(\'--height\', type=int, default=1024)\n    parser.add_argument(\'--width\', type=int, default=768)\n    parser.add_argument(\'--parts\', type=Path, nargs=\'+\')\n    args = parser.parse_args()\n    data_root = args.data_root or locate_dataset()\n    if args.stage == \'build\':\n        build_part(data_root, args.output, args.part, args.n_parts, args.workers, args.height, args.width)\n    else:\n        assemble_parts(args.parts or [], data_root, args.output, args.height, args.width)\n\n\nif __name__ == \'__main__\':\n    main()\n'
(PROJECT / "mammo_reliability" / 'cache_parts.py').write_text(SOURCE)


In [ ]:
subprocess.run([sys.executable, '-m', 'mammo_reliability.cache_parts', 'build', '--output', '/kaggle/working/rsna_cache_part_1', '--part', '1', '--n-parts', '2', '--workers', '4'], cwd=PROJECT, check=True)
